# 02 · Uplift modelling: estimating *individual* treatment effects
Five ways to estimate the effect for each user (CATE), compared with two baselines: a **response model**
(the common shortcut of targeting likely visitors) and **random** targeting. All models are trained on 70% of users
and evaluated on a 30% holdout with **Qini / uplift curves**, the standard way to evaluate uplift models on randomised data,
with bootstrap confidence intervals. **Runtime: CPU**

In [ ]:
# ---------- Setup: run this cell first (CPU runtime is enough) ----------
import os, sys, subprocess, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("UP_SMOKE") == "1"          # local smoke test on synthetic data
try:
    import google.colab  
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if SMOKE:
    PROJECT_DIR = os.environ["UP_PROJECT_DIR"]
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import glob
    hits = sorted(glob.glob("/content/drive/MyDrive/**/uplift/src/uplift/data.py", recursive=True), key=len)
    assert hits, "Upload the 'uplift' folder to your Google Drive first."
    PROJECT_DIR = str(Path(hits[0]).parents[2])                     # found automatically
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate"], check=True)
else:   # local Jupyter / VS Code: notebook lives in uplift/notebooks/
    PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, f"{PROJECT_DIR}/src")
ART, FIG = f"{PROJECT_DIR}/artifacts", f"{PROJECT_DIR}/reports/figures"
DATA = "/content/data" if ON_COLAB else f"{PROJECT_DIR}/data"
for d in (ART, FIG, DATA): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
from uplift.data import FEATURES, load_criteo, make_synthetic
from uplift import experiment as E, learners as L, metrics as M, causal as C
SEED = 42
df = make_synthetic(60_000) if SMOKE else load_criteo(DATA, sample_frac=0.10)   # 10% sample ~ 1.4M users
TREAT_SHARE = 0.85                                                              # by design of the experiment
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()
def save_json(o, name):
    json.dump(o, open(f"{ART}/{name}.json", "w"), indent=2, default=float)
def load_json(name):
    return json.load(open(f"{ART}/{name}.json"))
print(f"{len(df):,} users | treated share {df.treatment.mean():.3f} | visit {df.visit.mean():.4f} | conversion {df.conversion.mean():.5f}")

In [ ]:
from sklearn.model_selection import train_test_split
tr, te = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df.treatment * 2 + df.visit)
Xtr, Xte = tr[FEATURES].values, te[FEATURES].values
OUTCOME = "visit"
models, scores, fit_min = {}, {}, {}
for Cls in L.ALL_LEARNERS:
    t0 = time.time()
    kw = {"propensity": TREAT_SHARE} if Cls in (L.XLearner, L.DRLearner, L.TransformedOutcome) else {}
    m = Cls(seed=SEED, **kw).fit(Xtr, tr.treatment.values, tr[OUTCOME].values)
    models[m.name], scores[m.name] = m, m.predict(Xte); fit_min[m.name] = (time.time() - t0) / 60
    print(f"{m.name:28s} {fit_min[m.name]:.1f} min")

## 1. Qini curves (holdout)

In [ ]:
yte, tte = te[OUTCOME].values, te.treatment.values
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for name, s in scores.items():
    x, q = M.qini_curve(yte, tte, s); step = max(1, len(x) // 400)
    ax[0].plot(x[::step], q[::step], label=f"{name} ({M.qini_coefficient(yte, tte, s):.3f})",
               ls="--" if "Random" in name else "-")
    x, u = M.uplift_curve(yte, tte, s)
    ax[1].plot(x[::step], u[::step], label=name, ls="--" if "Random" in name else "-")
ax[0].set_title("Qini curve (legend: Qini coefficient)"); ax[0].set_xlabel("fraction of users targeted"); ax[0].set_ylabel("incremental visits")
ax[1].set_title("Uplift curve"); ax[1].set_xlabel("fraction of users targeted")
ax[0].legend(fontsize=8); savefig("02_qini_curves")

## 2. Model comparison with bootstrap 95% CIs

In [ ]:
NB = 30 if SMOKE else 100
rows = []
for name, s in scores.items():
    mean, lo, hi = M.bootstrap(yte, tte, s, M.qini_coefficient, n_boot=NB, seed=SEED)
    rows.append({"model": name, **M.summary(yte, tte, s), "qini_ci_low": lo, "qini_ci_high": hi, "fit_minutes": fit_min[name]})
comp = pd.DataFrame(rows).set_index("model").sort_values("qini_coef", ascending=False)
comp.to_csv(f"{ART}/model_comparison_{OUTCOME}.csv"); comp.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
c = comp.sort_values("qini_coef")
ax.barh(c.index, c.qini_coef, xerr=[c.qini_coef - c.qini_ci_low, c.qini_ci_high - c.qini_coef], color="#4C72B0", capsize=3)
ax.axvline(0, c="k", lw=1); ax.set_xlabel("Qini coefficient (95% bootstrap CI)"); ax.set_title(f"Holdout ranking quality ({OUTCOME})")
savefig("02_model_comparison")

## 3. Observed uplift by predicted-uplift decile (best model)

In [ ]:
BEST = comp.drop(index=[i for i in comp.index if "Random" in i or "Response" in i]).index[0]
dec = M.decile_table(yte, tte, scores[BEST]); dec_resp = M.decile_table(yte, tte, scores["Response model (baseline)"])
fig, ax = plt.subplots(figsize=(8, 3.8)); w = 0.4
ax.bar(np.arange(10) - w/2, dec.observed_uplift, w, label=BEST); ax.bar(np.arange(10) + w/2, dec_resp.observed_uplift, w, label="Response model")
ax.axhline(ate_all := yte[tte == 1].mean() - yte[tte == 0].mean(), c="r", ls="--", lw=1, label="average effect")
ax.set_xticks(range(10), range(1, 11)); ax.set_xlabel("decile of predicted score (1 = top)"); ax.set_ylabel("observed uplift")
ax.legend(); ax.set_title("Observed treatment effect per decile"); savefig("02_deciles"); dec.round(4)

## 4. Profit-optimal targeting
Illustrative economics (stated assumptions): each incremental visit is worth **v** and showing ads to a user costs **c**.
We choose the targeting fraction on one half of the holdout and report the profit on the other half, to avoid optimistic bias.

In [ ]:
VALUE, COST = 1.00, 0.005          # value per incremental visit, cost per targeted user (assumptions)
half = np.random.default_rng(SEED).random(len(yte)) < 0.5
pol = {}
for name in [BEST, "Response model (baseline)", "Random targeting"]:
    sel = M.policy_curve(yte[half], tte[half], scores[name][half], VALUE, COST)
    k = sel.loc[sel.profit_per_1000.idxmax(), "target_frac"]
    ev = M.policy_curve(yte[~half], tte[~half], scores[name][~half], VALUE, COST, grid=[k, 1.0])
    pol[name] = {"chosen_target_frac": k, "profit_per_1000 (chosen)": ev.profit_per_1000.iloc[0],
                 "profit_per_1000 (target everyone)": ev.profit_per_1000.iloc[1]}
pol = pd.DataFrame(pol).T; pol.to_csv(f"{ART}/policy.csv"); pol.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name in [BEST, "Response model (baseline)", "Random targeting"]:
    pc = M.policy_curve(yte, tte, scores[name], VALUE, COST)
    ax.plot(pc.target_frac, pc.profit_per_1000, marker="o", ms=3, label=name)
ax.axhline(0, c="k", lw=1); ax.set_xlabel("fraction of users targeted"); ax.set_ylabel("profit per 1,000 users")
ax.set_title(f"Targeting policy (value/visit={VALUE}, cost/user={COST})"); ax.legend(); savefig("02_policy")

## 5. Best two learners vs baselines

In [ ]:
conv = {}
for Cls in [L.XLearner, L.DRLearner, L.TLearner, L.ResponseModel, L.RandomScore]:
    kw = {"propensity": TREAT_SHARE} if Cls in (L.XLearner, L.DRLearner) else {}
    m = Cls(seed=SEED, **kw).fit(Xtr, tr.treatment.values, tr.conversion.values)
    s = m.predict(Xte)
    conv[m.name] = {**M.summary(te.conversion.values, tte, s), **dict(zip(["qini_boot_mean", "qini_ci_low", "qini_ci_high"],
                    M.bootstrap(te.conversion.values, tte, s, n_boot=NB, seed=SEED)))}
conv = pd.DataFrame(conv).T.sort_values("qini_coef", ascending=False); conv.to_csv(f"{ART}/model_comparison_conversion.csv"); conv.round(4)

In [ ]:
save_json({"best_model": BEST, "outcome": OUTCOME, "n_train": len(tr), "n_test": len(te), "value": VALUE, "cost": COST}, "modeling")
print("saved")